In [1]:
from bertopic import BERTopic
import json
import time

from nltk.tokenize import word_tokenize
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from itertools import combinations

In [2]:
K_RANGE = list(range(3, 20))
TOP_N = 10

In [3]:
try:
    with open('../dataProcessed/nurseNotes.json', 'r') as file:
        nurse_notes = json.load(file)
    print("File loaded successfully.")
    
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")

File loaded successfully.


In [4]:
# Make utility function to get coherence. - gensim implementation is also used in OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/coherence_metrics.py)
def get_coherence_score(topics, tokenized_texts, dictionary, coherence_type):
    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_texts,
        dictionary=dictionary,
        coherence=coherence_type # either 'c_npmi' or 'c_v'
    )
    return coherence_model.get_coherence()

# Make utility function to get diversity (always 10 words) - adapted from OCTIS (https://github.com/MIND-Lab/OCTIS/blob/master/octis/evaluation_metrics/diversity_metrics.py)
def get_diversity_score(topics):
    if len(topics) <= 0:
        return 0.0
    unique_words = set()
    for topic in topics:
        unique_words.update(topic[:10])
    return len(unique_words) / (10 * len(topics))

# Redundancy calculation. Higher the better, as closer to 1 means non-overlapping topics (inverted score). Adapted from https://aclanthology.org/2024.acl-long.11/.
def compute_topic_redundancy(topic_word_distributions, top_n=10):
    redundancy_scores = []
    for topic1, topic2 in combinations(topic_word_distributions, 2):
        overlap = len(set(topic1[:top_n]).intersection(set(topic2[:top_n])))
        redundancy = overlap / top_n
        redundancy_scores.append(redundancy)
    average_redundancy = sum(redundancy_scores) / len(redundancy_scores)
    return 1 - average_redundancy

In [5]:
def bertopic_analysis(texts):
    tokenized_texts = [word_tokenize(text.lower()) for text in texts]
    dictionary = Dictionary(tokenized_texts)

    print(f"Number of texts: {len(texts)}")

    start = time.time() 
    topic_model = BERTopic(
        embedding_model='sentence-transformers/all-MiniLM-L6-v2',
    )
    _topics, _probs = topic_model.fit_transform(texts)
    cluster_topics = list(topic_model.get_topic_info()['Representation'])
    cluster_topics = [topic[:TOP_N] for topic in cluster_topics]
    end = time.time()

    coherence = get_coherence_score(cluster_topics, tokenized_texts, dictionary, 'c_v')
    diversity = get_diversity_score(cluster_topics)
    redundancy = compute_topic_redundancy(cluster_topics)

    print(f"Coherence: {coherence}")
    print(f"Diversity: {diversity}")
    print(f"Inverse Redundancy: {redundancy}")
    print(f"Time (seconds): {end-start}")

    print("----- Cluster Topics -----")
    for cluster_topic in cluster_topics:
        print(cluster_topic)
        
    print(f"Number of Topics: {len(cluster_topics)}")

In [6]:
all_texts = []
for key in nurse_notes.keys():
    print(f"-----------{key}-----------")
    bertopic_analysis(nurse_notes[key]['Note'])
    all_texts.extend(nurse_notes[key]['Note'])

-----------P1-----------
Number of texts: 603


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6951903613069127
Diversity: 0.535
Inverse Redundancy: 0.9121052631578948
Time (seconds): 8.359400033950806
----- Cluster Topics -----
['to', 'plan', 'of', 'and', 'the', 'care', 'all', 'continue', 'no', 'on']
['sleeping', 'settled', 'bed', 'voiced', 'on', 'to', 'checks', 'post', 'medications', 'continued']
['eye', 'care', 'morning', 'as', 'medication', 'drops', 'assisted', 'given', 'usual', 'choice']
['checks', 'safety', 'on', 'continued', 'comfortable', 'meds', 'due', 'taken', 'settled', 'asleep']
['relaxed', 'by', 'staff', 'adl', 'her', 'content', 'taken', 'and', 'appears', 'with']
['rollator', 'mobilising', 'good', 'given', 'with', 'form', 'steroids', 'complaints', 'well', 'appears']
['toiletting', 'ongoing', 'asleep', 'self', 'comfortable', 'checks', 'resident', 'toileting', 'peacefully', 'required']
['mobile', 'restaurant', 'usual', 'her', 'attended', 'independent', 'form', 'in', 'needs', 'due']
['peaceful', 'toiletting', 'ongoing', 'asleep', 'self', 'checks', 'residen

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6958502990025812
Diversity: 0.638095238095238
Inverse Redundancy: 0.9423809523809524
Time (seconds): 9.88048005104065
----- Cluster Topics -----
['plan', 'hospital', 'care', 'for', 'no', 'resident', 'see', 'of', 'in', 'mdt']
['checks', 'night', 'safety', 'well', 'comfortable', 'concerns', 'resident', 'at', 'check', 'needs']
['toileted', 'and', 'settled', 'tts', 'comfortably', 'night', 'medications', 'given', 'hoisted', 'bed']
['baseline', 'prescribed', 'took', 'wash', 'wheelchair', 'am', 'electric', 'transferred', 'this', 'skin']
['received', 'all', 'appeared', 'till', 'needs', 'charted', 'resident', 'ensured', 'complaint', 'due']
['adls', 'adequate', 'new', 'pu', 'appears', 'as', 'charted', 'given', 'intake', 'with']
['with', 'as', 'charted', 'diet', 'resident', 'weight', 'meds', 'good', 'please', 'protein']
['daughter', 'of', 'gp', 'vomiting', 'resident', 'urine', 'hospital', 'and', 'to', 'vomit']
['laxative', 'bno', 'took', 'wash', 'with', 'skin', 'morning', 'assisted',

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.7589631821677817
Diversity: 0.5136363636363637
Inverse Redundancy: 0.9173160173160173
Time (seconds): 12.741083860397339
----- Cluster Topics -----
['her', 'in', 'the', 'is', 'and', 'she', 'to', 'resident', 'bed', 'care']
['adls', 'compliant', 'needed', 'maintained', 'as', 'settled', 'for', 'night', 'safety', 'charted']
['care', 'eye', 'drinking', 'new', 'no', 'eating', 'and', 'done', 'issues', 'given']
['compliant', 'adls', 'having', 'needed', 'maintained', 'as', 'night', 'settled', 'is', 'safety']
['paracetamol', 'pain', 'prn', 'by', 'vaccine', 'administered', 'gp', 'to', 'on', 'and']
['on', 'comfortable', 'toilet', 'nocte', 'early', 'asleep', 'checks', 'bed', 'place', 'appears']
['morning', 'her', 'given', 'in', 'care', 'enjoyed', 'form', 'breakfast', 'this', 'personal']
['to', 'and', 'her', 'the', 'alarm', 'of', 'staff', 'insitu', 'remains', 'bed']
['toileting', 'going', 'on', 'checks', 'needs', 'place', 'due', 'safety', 'mat', 'taken']
['complaint', 'voiced', 'appeare

In [7]:
bertopic_analysis(all_texts)

Number of texts: 12373


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Coherence: 0.6122151267626972
Diversity: 0.30762463343108504
Inverse Redundancy: 0.9711195445920303
Time (seconds): 163.2856330871582
----- Cluster Topics -----
['to', 'the', 'of', 'and', 'taken', 'done', 'today', 'resident', 'doctor', 'given']
['continues', 'report', 'planned', 'changes', 'hourly', 'or', 'well', 'no', 'sleep', 'sleeping']
['compliant', 'needed', 'adls', 'maintained', 'for', 'night', 'settled', 'as', 'safety', 'charted']
['sunday', 'mobile', 'required', 'activities', 'lunch', 'restaurant', 'prayers', 'form', 'attending', 'content']
['paracetamol', 'pain', 'prn', 'complained', 'requested', 'shoulder', 'hip', '00', 'back', 'regular']
['baseline', 'mobility', 'wash', 'prescribed', 'dining', 'took', 'am', 'unit', 'meals', 'be']
['ensured', 'till', 'received', 'noted', 'observed', 'kept', 'time', 'from', 'him', 'met']
['coughing', 'cough', 'exputex', 'chesty', 'occasional', 'prn', 'chest', 'nebs', 'syrup', 'intermittent']
['personal', 'entry', 'mood', 'mobile', 'attending',